# 🔄 F1 Silver — RESULTS — SCD TYPE 2 DEMO

This notebook is a **separate SCD Type 2 demonstration** for the existing Results incremental process.

### Existing SCD Type 1
Your original SCD Type 1 notebook is **not changed**. It continues to use:

```text
Matched     → UPDATE
Not Matched → INSERT
```

### SCD Type 2 in this notebook

```text
New record       → INSERT
Unchanged record → NO ACTION
Changed record   → EXPIRE old + INSERT new version
```

SCD2 technical columns:

- `effective_start_date`
- `effective_end_date`
- `is_current`
- `record_hash`

**Business key:** `result_id`

The SCD2 demo writes to a separate table:

`formula1_<env>.silver.results_scd2_demo`


In [0]:
# Databricks notebook source
# Environment widget

dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

print("Environment:", env)


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = f"formula1_{env}"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

def bronze_table(name):
    return f"{CATALOG}.{BRONZE}.{name}"

def silver_table(name):
    return f"{CATALOG}.{SILVER}.{name}"

print("Bronze:", bronze_table("results"))
print("SCD2 Target:", silver_table("results_scd2_demo"))


## 1. Read Bronze Results

This follows the same source table used by the existing Results notebook.


In [0]:
results = spark.table(bronze_table("results"))

print("Bronze rows:", results.count())
results.printSchema()


## 2. Identify the Incremental Batch

The existing Results notebook uses `ingestion_timestamp` as the watermark and `result_id` as the MERGE/business key.

For this SCD2 demo, the watermark is maintained by the **SCD2 target itself**.

That is important because the SCD2 table contains historical versions, so we only use its maximum `ingestion_timestamp` to identify newly arrived Bronze records.


In [0]:
scd2_target_name = silver_table("results_scd2_demo")

if not spark.catalog.tableExists(scd2_target_name):
    last_ts = None
else:
    last_ts = (
        spark.table(scd2_target_name)
        .agg(F.max("ingestion_timestamp").alias("max_ts"))
        .first()["max_ts"]
    )

print("Last SCD2 ingestion_timestamp:", last_ts)

results_batch = (
    results
    if last_ts is None
    else results.filter(F.col("ingestion_timestamp") > F.lit(last_ts))
)

print("Rows selected for SCD2:", results_batch.count())


## 3. Clean the Selected Batch

The transformation logic follows the existing Results notebook.


In [0]:
results_clean = (
    results_batch
    .filter(F.col("result_id").isNotNull())
    .filter(F.col("race_id").isNotNull())
    .filter(F.col("driver_id").isNotNull())
    .filter(F.col("constructor_id").isNotNull())
    .filter(F.col("points").isNull() | (F.col("points") >= 0))
    .dropDuplicates(["result_id"])
    .withColumn("position_text_clean", F.trim("position_text"))
    .withColumn("position_order_int", F.col("position_order").cast("int"))
    .withColumn("points_clean", F.coalesce("points", F.lit(0.0)))
    .withColumn(
        "finish_category",
        F.when(F.col("position") == 1, "Winner")
         .when(F.col("position").isin(2, 3), "Podium")
         .when(F.col("position").between(4, 10), "Points Finish")
         .otherwise("Outside Points")
    )
    .withColumn("silver_processed_timestamp", F.current_timestamp())
)

print("Clean incremental rows:", results_clean.count())
display(results_clean.limit(20))


## 4. Create the SCD2 Source

### Why `record_hash`?

We need to distinguish:

```text
Same result_id + same data
        → unchanged

Same result_id + different data
        → changed
```

Technical timestamps are **excluded** from the hash. Otherwise every new ingestion could incorrectly look like a business-data change.


In [0]:
business_columns = [
    c for c in results_clean.columns
    if c not in [
        "ingestion_timestamp",
        "silver_processed_timestamp",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "record_hash"
    ]
]

print("Columns used for change detection:")
print(business_columns)

scd2_source = (
    results_clean
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))
                    for c in business_columns
                ]
            ),
            256
        )
    )
)

display(
    scd2_source
    .select("result_id", "points", "record_hash")
    .limit(20)
)


## 5. Initial SCD2 Load

For the first run:

```text
Every source record
       ↓
is_current = true
effective_end_date = NULL
```


In [0]:
if not spark.catalog.tableExists(scd2_target_name):

    batch_timestamp = spark.sql("SELECT current_timestamp()").first()[0]

    scd2_initial = (
        scd2_source
        .withColumn("effective_start_date", F.lit(batch_timestamp).cast("timestamp"))
        .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )

    (
        scd2_initial.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(scd2_target_name)
    )

    print("SCD2 initial load completed.")

else:
    print("SCD2 target already exists.")


## 6. Detect New / Changed / Unchanged Records

Only the **current** Silver version is compared.

```text
Incoming result_id
        ↓
Current Silver record?
   ┌────┴────┐
  NO        YES
  ↓          ↓
 NEW      Compare hash
             ↓
       ┌─────┴─────┐
     SAME         DIFFERENT
       ↓             ↓
  UNCHANGED       CHANGED
```


In [0]:
current_target = (
    spark.table(scd2_target_name)
    .filter(F.col("is_current") == True)
    .select(
        "result_id",
        "record_hash"
    )
)

comparison = (
    scd2_source.alias("s")
    .join(
        current_target.alias("t"),
        F.col("s.result_id") == F.col("t.result_id"),
        "left"
    )
    .select(
        F.col("s.*"),
        F.col("t.record_hash").alias("target_hash")
    )
)

new_records = comparison.filter(
    F.col("target_hash").isNull()
)

changed_records = comparison.filter(
    F.col("target_hash").isNotNull()
    & (F.col("record_hash") != F.col("target_hash"))
)

unchanged_records = comparison.filter(
    F.col("target_hash").isNotNull()
    & (F.col("record_hash") == F.col("target_hash"))
)

print("New records      :", new_records.count())
print("Changed records  :", changed_records.count())
print("Unchanged records:", unchanged_records.count())


## 7. Build SCD2 Staging Data

This is the key SCD2 technique.

For a changed record we create **two staging rows**:

### Row 1 — EXPIRE

```text
merge_result_id = existing result_id
action = EXPIRE
```

This row matches the current target record and expires it.

### Row 2 — INSERT

```text
merge_result_id = NULL
action = INSERT
```

Because the merge key is NULL, it does not match an existing target row and therefore inserts the new version.

This lets us perform **expire + insert in one Delta MERGE**.


In [0]:
# One timestamp for the whole incoming batch
batch_timestamp = spark.sql("SELECT current_timestamp()").first()[0]

# Columns in the existing SCD2 target
target_columns = spark.table(scd2_target_name).columns

# ------------------------------------------------------------
# 1. EXPIRE staging rows
# ------------------------------------------------------------

expire_stage = (
    changed_records
    .select(
        F.col("result_id").alias("merge_result_id")
    )
    .withColumn("action", F.lit("EXPIRE"))
)

# Add all target columns with NULL values.
# They are not used for the update except effective_end_date/is_current.
for field in spark.table(scd2_target_name).schema.fields:
    if field.name not in expire_stage.columns:
        expire_stage = expire_stage.withColumn(
            field.name,
            F.lit(None).cast(field.dataType)
        )

expire_stage = (
    expire_stage
    .withColumn("effective_end_date", F.lit(batch_timestamp).cast("timestamp"))
    .withColumn("is_current", F.lit(False))
)

# ------------------------------------------------------------
# 2. INSERT staging rows for changed records
# ------------------------------------------------------------

insert_changed_stage = (
    changed_records
    .withColumn("effective_start_date", F.lit(batch_timestamp).cast("timestamp"))
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn("merge_result_id", F.lit(None).cast("int"))
    .withColumn("action", F.lit("INSERT"))
)

# ------------------------------------------------------------
# 3. INSERT staging rows for brand-new records
# ------------------------------------------------------------

insert_new_stage = (
    new_records
    .withColumn("effective_start_date", F.lit(batch_timestamp).cast("timestamp"))
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn("merge_result_id", F.lit(None).cast("int"))
    .withColumn("action", F.lit("INSERT"))
)

# Align the three DataFrames to one schema.
stage_columns = (
    [f.name for f in spark.table(scd2_target_name).schema.fields]
    + ["merge_result_id", "action"]
)

def align_stage(df):
    for field in spark.table(scd2_target_name).schema.fields:
        if field.name not in df.columns:
            df = df.withColumn(
                field.name,
                F.lit(None).cast(field.dataType)
            )
    return df.select(*stage_columns)

scd2_stage = (
    align_stage(expire_stage)
    .unionByName(align_stage(insert_changed_stage))
    .unionByName(align_stage(insert_new_stage))
)

print("SCD2 staging rows:", scd2_stage.count())

display(
    scd2_stage.select(
        "merge_result_id",
        "result_id",
        "points",
        "action",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
)


## 8. Single Delta MERGE — SCD Type 2

### MERGE condition

```text
target.result_id = staging.merge_result_id
```

Therefore:

- `EXPIRE` row → matches current target → UPDATE
- `INSERT` row → `merge_result_id = NULL` → does not match → INSERT

Unchanged records are not placed into the staging DataFrame.


In [0]:
target = DeltaTable.forName(
    spark,
    scd2_target_name
)

# Automatically build the INSERT mapping from the actual target schema.
# This avoids hard-coding every Results column.
insert_values = {
    field.name: f"s.{field.name}"
    for field in spark.table(scd2_target_name).schema.fields
}

(
    target.alias("t")
    .merge(
        scd2_stage.alias("s"),
        "t.result_id = s.merge_result_id"
    )
    .whenMatchedUpdate(
        condition="s.action = 'EXPIRE'",
        set={
            "effective_end_date": "s.effective_end_date",
            "is_current": "s.is_current"
        }
    )
    .whenNotMatchedInsert(
        condition="s.action = 'INSERT'",
        values=insert_values
    )
    .execute()
)

print("SCD2 MERGE completed.")


## 9. Validate SCD2

Expected behavior:

### Same record on second run

```text
New       = 0
Changed   = 0
Unchanged > 0

No duplicate version created.
```

### Changed record

```text
Old version → is_current = false
New version → is_current = true
```


In [0]:
scd2_result = spark.table(scd2_target_name)

print("Total SCD2 rows:", scd2_result.count())

display(
    scd2_result
    .orderBy("result_id", "effective_start_date")
)


In [0]:
# Only one current record is allowed for each result_id.

duplicate_current = (
    scd2_result
    .filter(F.col("is_current") == True)
    .groupBy("result_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate current records:")
display(duplicate_current)


## 10. Check One Result's History

Change the `result_id` below for your classroom demonstration.

If the record changed, you should see:

```text
old value → current = false → end date populated
new value → current = true  → end date NULL
```


In [0]:
demo_result_id = 25004

display(
    scd2_result
    .filter(F.col("result_id") == demo_result_id)
    .select(
        "result_id",
        "points",
        "position",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .orderBy("effective_start_date")
)


In [0]:
%sql
select count(*) from formula1_dev.silver.results_scd2_demo

In [0]:
%sql
select * from formula1_dev.silver.results_scd2_demo
order by result_id desc